# L46 · Mel 频率尺度 — 人耳对数感知，mel = 2595·log₁₀(1+f/700)，三角滤波器

**目标**：实现 Hz↔Mel 转换和三角滤波器组，理解为什么 Mel 特征比线性频谱更适合 ASR。

🔗 Aurora 连接：`aurora.audio.mel.hz_to_mel()` 和 `aurora.audio.mel.mel_filterbank()`

人耳对频率的感知是对数的，而非线性的：100 Hz 与 200 Hz 之间的差异听起来和 3000 Hz 与 3100 Hz 之间的差异完全不同。Mel 标度把这种感知非线性「拉直」，让相同 Mel 间隔对应相同的主观音调差异。三角滤波器组在 Mel 域均匀分布，转回 Hz 域后自动形成低频密、高频疏的对数间隔，正是 ASR 声学特征的基础。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Mel 标度公式

Mel 标度由 Stevens、Volkmann、Newman（1937）提出，常用近似公式为：

```
f_mel = 2595 * log10(1 + f_hz / 700)
```

700 Hz 的音调对应约 783 Mel。注意：
- 低频区间（0–1000 Hz）：每 100 Hz 对应约 100+ Mel，分辨率高
- 高频区间（3000–4000 Hz）：每 100 Hz 对应约 30 Mel，分辨率低

逆变换（Mel → Hz）：
```
f_hz = 700 * (10^(f_mel / 2595) - 1)
```

In [ ]:
# 演示 Mel 标度的对数特性
freqs = np.array([100, 200, 500, 1000, 2000, 4000, 8000])
mels_approx = 2595 * np.log10(1 + freqs / 700)

print(f"{'Hz':>8}  {'Mel':>8}  {'Delta_Mel':>10}")
for i, (f, m) in enumerate(zip(freqs, mels_approx)):
    delta = f"{mels_approx[i] - mels_approx[i-1]:.1f}" if i > 0 else "—"
    print(f"{f:>8}  {m:>8.1f}  {delta:>10}")

## 2. 三角滤波器组的构造

构造步骤：
1. 在 Mel 域从 0 到 `hz_to_mel(sr/2)` 均匀取 `n_mels + 2` 个中心点（含左右边界）
2. 将这 `n_mels + 2` 个 Mel 值转回 Hz，得到非均匀间隔的中心频率
3. 把每个 Hz 中心频率映射到最近的 FFT bin 索引
4. 第 `m` 个三角滤波器：在 `[center[m-1], center[m]]` 线性升至 1，在 `[center[m], center[m+1]]` 线性降至 0

每个滤波器的形状（在 Hz 域）是不对称三角形——低频滤波器窄，高频滤波器宽，天然匹配人耳分辨率。

In [ ]:
# 可视化：Mel 域均匀 → Hz 域非均匀
sr = 16000
n_mels = 10
mel_max = 2595 * np.log10(1 + (sr / 2) / 700)
mel_points = np.linspace(0, mel_max, n_mels + 2)
hz_points = 700 * (10 ** (mel_points / 2595) - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(mel_points, np.zeros(n_mels+2), 'o')
axes[0].set_title('Mel 域：均匀间隔')
axes[0].set_xlabel('Mel')
axes[1].plot(hz_points, np.zeros(n_mels+2), 'o')
axes[1].set_title('Hz 域：对数间隔（低密高疏）')
axes[1].set_xlabel('Hz')
plt.tight_layout()
plt.show()

## 3. Filterbank 矩阵与功率谱相乘

`mel_filterbank()` 返回 shape `(n_mels, n_fft//2+1)` 的矩阵，每行是一个三角滤波器在 FFT bin 轴上的权重。

```
fb = mel_filterbank(40, 512, 16000)   # shape (40, 257)
power_spec = np.abs(X)**2             # shape (257,)，单帧功率谱
mel_energy = fb @ power_spec          # shape (40,)，Mel 能量
```

`@` 一次矩阵乘法完成 40 个三角积分。对 `mel_energy` 取 log 即得 log-Mel 特征，再做 DCT 即得 MFCC（Day 5）。

In [ ]:
# 演示矩阵乘法：随机功率谱 -> Mel 能量
# （真实 filterbank 在编码任务后实现）
np.random.seed(42)
n_fft = 512
fake_power = np.random.rand(n_fft // 2 + 1)  # shape (257,)
# 占位：用全 1 filterbank 演示形状
fb_placeholder = np.ones((40, n_fft // 2 + 1)) / (n_fft // 2 + 1)
mel_e = fb_placeholder @ fake_power
print(f"功率谱 shape: {fake_power.shape}")
print(f"filterbank shape: {fb_placeholder.shape}")
print(f"Mel 能量 shape: {mel_e.shape}")

## 4. ✏️ 实现 `hz_to_mel(f)`

**推理路线**：
1. 公式直接套用：`2595 * np.log10(1 + f / 700)`，numpy 广播支持标量和数组

**参考输入输出**：
- `hz_to_mel(0)` → `0.0`
- `hz_to_mel(700)` → `≈ 782.7`
- `hz_to_mel(np.array([0, 700, 8000]))` → `array([0., 782.7, 2840.0])`

<details><summary>点击查看参考实现</summary>

```python
def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, dtype=float) / 700)
```

</details>

In [ ]:
def hz_to_mel(f):
    # ✏️ TODO: 套用公式 2595 * log10(1 + f/700)
    # 提示：用 np.log10，支持标量和数组
    pass

In [ ]:
assert abs(hz_to_mel(700) - 782.7) < 0.5, f"hz_to_mel(700) = {hz_to_mel(700):.2f}"
assert hz_to_mel(0) == 0.0, "hz_to_mel(0) 应为 0"
result = hz_to_mel(np.array([0, 700, 8000]))
assert result.shape == (3,), "数组输入应返回同 shape"
print(f"hz_to_mel(700) = {hz_to_mel(700):.2f}  ✅")
print(f"hz_to_mel([0, 700, 8000]) = {hz_to_mel(np.array([0, 700, 8000]))}  ✅")

## 5. ✏️ 实现 `mel_filterbank(n_mels, n_fft, sr)`

**推理路线**：
1. Mel 域取 `n_mels + 2` 个均匀点（含 0 和 `hz_to_mel(sr/2)` 两端），用 `np.linspace`
2. 将所有 Mel 点转回 Hz：`700 * (10**(mel_points / 2595) - 1)`
3. 把 Hz 映射到 FFT bin 索引：`np.floor(hz_points / (sr / n_fft) * ... )`——精确公式为 `np.floor((n_fft + 1) * hz_points / sr).astype(int)`
4. 对每个滤波器 `m`（0 到 n_mels-1），在 `[bin[m], bin[m+1], bin[m+2]]` 区间构造上升/下降斜坡：
   - 上升段：`(k - bin[m]) / (bin[m+1] - bin[m])` for k in `[bin[m], bin[m+1]]`
   - 下降段：`(bin[m+2] - k) / (bin[m+2] - bin[m+1])` for k in `[bin[m+1], bin[m+2]]`

**参考输入输出**：
- `mel_filterbank(40, 512, 16000).shape` → `(40, 257)`
- 所有元素 `>= 0`，每行峰值为 1
- `mel_filterbank(40, 512, 16000)[0, :10]`：前几个 bin 有非零权重（第 0 个滤波器覆盖低频）

<details><summary>点击查看参考实现</summary>

```python
def mel_filterbank(n_mels, n_fft, sr):
    mel_min = 0.0
    mel_max = hz_to_mel(sr / 2)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    bin_idx = np.floor((n_fft + 1) * hz_points / sr).astype(int)

    n_bins = n_fft // 2 + 1
    fb = np.zeros((n_mels, n_bins))
    for m in range(n_mels):
        lo, ctr, hi = bin_idx[m], bin_idx[m+1], bin_idx[m+2]
        for k in range(lo, ctr):
            fb[m, k] = (k - lo) / (ctr - lo) if ctr != lo else 0
        for k in range(ctr, hi):
            fb[m, k] = (hi - k) / (hi - ctr) if hi != ctr else 0
    return fb
```

</details>

In [ ]:
def mel_filterbank(n_mels, n_fft, sr):
    # ✏️ TODO 步骤 1：Mel 域均匀取 n_mels+2 个点
    mel_min = 0.0
    mel_max = None  # ✏️ TODO: hz_to_mel(sr/2)
    mel_points = None  # ✏️ TODO: np.linspace(...)

    # ✏️ TODO 步骤 2：Mel 点转回 Hz
    hz_points = None  # ✏️ TODO: 700 * (10**(mel_points/2595) - 1)

    # ✏️ TODO 步骤 3：Hz 转 FFT bin 索引
    bin_idx = None  # ✏️ TODO: np.floor((n_fft+1) * hz_points / sr).astype(int)

    # ✏️ TODO 步骤 4：构造三角滤波器矩阵
    n_bins = n_fft // 2 + 1
    fb = np.zeros((n_mels, n_bins))
    for m in range(n_mels):
        lo, ctr, hi = None, None, None  # ✏️ TODO: bin_idx[m], bin_idx[m+1], bin_idx[m+2]
        # ✏️ TODO: 上升段 [lo, ctr)，下降段 [ctr, hi)
        pass
    return fb

In [ ]:
fb = mel_filterbank(40, 512, 16000)
assert fb.shape == (40, 257), f"shape 错误: {fb.shape}"
assert np.all(fb >= 0), "滤波器权重不应为负"
assert np.allclose(fb.max(axis=1), 1.0), "每个滤波器峰值应为 1"
print(f"filterbank shape: {fb.shape}  ✅")
print(f"所有权重 >= 0: {np.all(fb >= 0)}  ✅")
print(f"各行最大值均为 1: {np.allclose(fb.max(axis=1), 1.0)}  ✅")

## 6. 参数实验：可视化滤波器组

修改参数观察效果：
- `n_mels=40`（标准 ASR）vs `n_mels=80`（高分辨率）：滤波器数量翻倍，高频细节更丰富
- `sr=16000` vs `sr=8000`：采样率减半时最高覆盖频率从 8000 Hz 降至 4000 Hz，滤波器更集中在低频
- 图1（Hz 线性轴）：低频滤波器窄且密集，高频滤波器宽且稀疏——体现对数特性
- 图2（Mel 轴）：所有滤波器宽度相等，均匀分布

In [ ]:
fb40 = mel_filterbank(40, 512, 16000)
n_bins = 512 // 2 + 1
freq_axis = np.linspace(0, 16000 / 2, n_bins)
mel_axis = hz_to_mel(freq_axis)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 图1：Hz 线性轴
for m in range(40):
    axes[0].plot(freq_axis, fb40[m], alpha=0.5, linewidth=0.8)
axes[0].set_title('40 个 Mel 滤波器（Hz 线性轴）')
axes[0].set_xlabel('频率 (Hz)')
axes[0].set_ylabel('权重')
axes[0].set_xlim(0, 8000)

# 图2：Mel 轴
for m in range(40):
    axes[1].plot(mel_axis, fb40[m], alpha=0.5, linewidth=0.8)
axes[1].set_title('40 个 Mel 滤波器（Mel 轴）')
axes[1].set_xlabel('频率 (Mel)')
axes[1].set_ylabel('权重')

plt.suptitle('n_mels=40, n_fft=512, sr=16000', fontsize=12)
plt.tight_layout()
plt.show()
print("左图低频密/高频疏 → 右图均匀分布，体现 Mel 标度的对数特性")

## 本课小结

本课实现了 `hz_to_mel()` 和 `mel_filterbank()`，前者把线性 Hz 映射到感知对数 Mel 标度，后者构造 shape `(n_mels, n_fft//2+1)` 的三角权重矩阵。将该矩阵乘以 STFT 功率谱，一步得到 `(n_mels,)` 的 Mel 能量向量，与 `aurora.audio.mel` 模块中的实现等价。下节（Day 5）将对 log-Mel 能量做 DCT，提取最终的 MFCC 系数，完成声学特征流水线。